# Local SGI analysis

Run this notebook in VS Code with the project's `.venv` Python kernel. It reads the saved per-response SGI file, compares the 0.70-1.40 threshold grid and class-mean midpoints, and updates `ANALYSIS.md` and CSV exports directly in this repository. No Google Drive mount, embedding model or GPU is needed.

The source scores are local and ignored by Git. Embedding generation remains in the Colab notebook. Midpoint estimates and direction checks are descriptive and use the same labeled data for estimation and evaluation.


In [1]:
from pathlib import Path
import sys
import platform

print("Python:", sys.executable)
print("Kernel:", platform.release())

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
                    if (path / "scripts" / "analyze_sgi.py").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from scripts.analyze_sgi import run_analysis

results = run_analysis(PROJECT_ROOT)
print("Results saved to:", results["output_dir"])
print("Updated:", PROJECT_ROOT / "ANALYSIS.md")


Python: /home/maciej/github/Hallucination_detector/.venv/bin/python
Kernel: 6.6.87.2-microsoft-standard-WSL2


Results saved to: /home/maciej/github/Hallucination_detector/results/HaluEval/thresholds_0.70_1.40
Updated: /home/maciej/github/Hallucination_detector/ANALYSIS.md


In [2]:
from IPython.display import display, Markdown

display(Markdown("### Accuracy by threshold"))
display(results["accuracy"])
display(Markdown("### Accuracy change from threshold 1.00, percentage points"))
display(results["accuracy_change_pp"])
display(Markdown("### Class means and midpoint estimates"))
display(results["midpoint_estimates"])
display(Markdown("### Midpoint decisions"))
display(results["midpoint_metrics"])


### Accuracy by threshold

subset,qa,dialogue,summarization,all
threshold,,,,
0.70,0.61175,0.50305,0.50000,0.538267
0.75,0.64530,0.50625,0.50000,0.550517
0.80,0.67830,0.51385,0.50000,0.564050
0.85,0.70565,0.52055,0.50000,0.575400
0.90,0.70165,0.52485,0.50000,0.575500
0.95,0.66730,0.50750,0.50000,0.558267
1.00,0.63765,0.47935,0.50000,0.539000
1.05,0.60985,0.46415,0.50000,0.524667
1.10,0.59165,0.45705,0.49995,0.516217


### Accuracy change from threshold 1.00, percentage points

subset,qa,dialogue,summarization,all
threshold,,,,
0.70,-2.590,2.370,0.000,-0.073333
0.75,0.765,2.690,0.000,1.151667
0.80,4.065,3.450,0.000,2.505000
0.85,6.800,4.120,0.000,3.640000
0.90,6.400,4.550,0.000,3.650000
0.95,2.965,2.815,0.000,1.926667
1.00,0.000,0.000,0.000,0.000000
1.05,-2.780,-1.520,0.000,-1.433333
1.10,-4.600,-2.230,-0.005,-2.278333


### Class means and midpoint estimates

,subset,grounded_rows,hallucinated_rows,mean_sgi_grounded,mean_sgi_hallucinated,midpoint_threshold,class_mean_direction,outside_grid
0,qa,10000,10000,1.066184,0.885615,0.975899,below,False
1,dialogue,10000,10000,2.142929,1.057075,1.600002,below,True
2,summarization,10000,10000,2.500901,2.683187,2.592044,above,True
3,all,30000,30000,1.903338,1.541959,1.722648,below,True


### Midpoint decisions

,subset,threshold,rows,evaluated_rows,excluded_rows,tp,fp,tn,fn,accuracy,...,predicted_hallucination_rate,policy,prediction_direction,grounded_rows,hallucinated_rows,mean_sgi_grounded,mean_sgi_hallucinated,midpoint_threshold,class_mean_direction,outside_grid
0,qa,0.975899,20000,20000,0,6917,3901,6099,3083,0.65080,...,0.540900,fixed_below,below,10000,10000,1.066184,0.885615,0.975899,below,False
1,qa,0.975899,20000,20000,0,6917,3901,6099,3083,0.65080,...,0.540900,class_mean_direction,below,10000,10000,1.066184,0.885615,0.975899,below,False
2,dialogue,1.600002,20000,20000,0,9744,9904,96,256,0.49200,...,0.982400,fixed_below,below,10000,10000,2.142929,1.057075,1.600002,below,True
3,dialogue,1.600002,20000,20000,0,9744,9904,96,256,0.49200,...,0.982400,class_mean_direction,below,10000,10000,2.142929,1.057075,1.600002,below,True
4,summarization,2.592044,20000,20000,0,4842,6174,3826,5158,0.43340,...,0.550800,fixed_below,below,10000,10000,2.500901,2.683187,2.592044,above,True
5,summarization,2.592044,20000,20000,0,5158,3826,6174,4842,0.56660,...,0.449200,class_mean_direction,above,10000,10000,2.500901,2.683187,2.592044,above,True
6,all,1.722648,60000,60000,0,20044,20389,9611,9956,0.49425,...,0.673883,fixed_below,below,30000,30000,1.903338,1.541959,1.722648,below,True
7,all,1.722648,60000,60000,0,20044,20389,9611,9956,0.49425,...,0.673883,class_mean_direction,below,30000,30000,1.903338,1.541959,1.722648,below,True


In [3]:
display(Markdown(results["report"]))


# Experiment analysis

## Scope

SGI is evaluated on HaluEval QA, dialogue and summarization. This is not a full replication of the paper. Dialogue uses the conversation history as q; summarization uses a fixed instruction added by this project.

## Approach 1: fixed threshold grid

Predict hallucination when SGI < threshold; equality is grounded. The current grid is 0.70 to 1.40, inclusive, in steps of 0.05. Accuracy changes are measured against threshold 1.00 in percentage points. These results describe the saved labeled scores, not an independent test set.

| SGI threshold | QA accuracy | Dialogue accuracy | Summarization accuracy | All accuracy |
| ---: | ---: | ---: | ---: | ---: |
| 0.70 | 61.18% | 50.30% | 50.00% | 53.83% |
| 0.75 | 64.53% | 50.62% | 50.00% | 55.05% |
| 0.80 | 67.83% | 51.39% | 50.00% | 56.41% |
| 0.85 | 70.56% | 52.05% | 50.00% | 57.54% |
| 0.90 | 70.17% | 52.49% | 50.00% | 57.55% |
| 0.95 | 66.73% | 50.75% | 50.00% | 55.83% |
| 1.00 | 63.77% | 47.94% | 50.00% | 53.90% |
| 1.05 | 60.98% | 46.41% | 50.00% | 52.47% |
| 1.10 | 59.16% | 45.70% | 49.99% | 51.62% |
| 1.15 | 57.53% | 45.85% | 49.98% | 51.12% |
| 1.20 | 55.81% | 46.35% | 49.98% | 50.71% |
| 1.25 | 54.20% | 46.87% | 49.96% | 50.34% |
| 1.30 | 53.10% | 47.30% | 49.94% | 50.11% |
| 1.35 | 52.04% | 47.80% | 49.94% | 49.93% |
| 1.40 | 51.30% | 48.21% | 49.90% | 49.80% |

## Approach 2: midpoint of class means

For each task, threshold = (mean grounded SGI + mean hallucinated SGI) / 2. The pooled row is an additional summary, not a replacement for task-specific thresholds. Midpoints are evaluated at full precision and are not clipped to the grid. Means, thresholds and metrics use only finite SGI values with binary labels.

| Task | Mean SGI: grounded | Mean SGI: hallucinated | Midpoint | Accuracy: SGI < midpoint | Mean-based rule | Accuracy: mean-based rule |
| --- | ---: | ---: | ---: | ---: | --- | ---: |
| qa | 1.066184 | 0.885615 | 0.975899 | 65.08% | SGI < midpoint | 65.08% |
| dialogue | 2.142929 | 1.057075 | 1.600002 | 49.20% | SGI < midpoint | 49.20% |
| summarization | 2.500901 | 2.683187 | 2.592044 | 43.34% | SGI > midpoint | 56.66% |
| all | 1.903338 | 1.541959 | 1.722648 | 49.43% | SGI < midpoint | 49.43% |

The fixed-below column preserves the original SGI rule. The mean-based diagnostic flags scores above the midpoint when the hallucinated mean exceeds the grounded mean. Equality is grounded in both policies. This second policy changes the decision direction, not the score or the baseline AUROC. When the direction is below, the policies are identical.

The midpoint is a heuristic, not an optimal threshold. It ignores dispersion, extreme ratios, class priors and error costs. The means, threshold and diagnostic direction were estimated on the same data used for evaluation, so these are exploratory results. Held-out performance remains unknown. Do not report the highest grid accuracy as validated test accuracy.

## Interpretation and next checks

The original baseline AUROC was 0.7399 for QA, 0.4745 for dialogue and 0.4084 for summarization using -SGI. The new cutoffs do not change that ranking. If the summarization midpoint lies outside the grid, it is still evaluated separately. The test asks whether a different cutoff helps, and whether the reversed mean order warrants a separate direction check. It does not establish why the ranking is reversed.

- [x] Implement and execute the 0.70-1.40 grid on saved scores.
- [x] Compute class means, midpoint thresholds and decision metrics.
- [ ] Review whether the midpoint improves accuracy, recall and false alarm rate against threshold 1.00.
- [ ] Inspect dispersion and extreme ratios, especially for dialogue.
- [ ] Select a threshold and direction on development pairs and evaluate once on separate test pairs.
- [ ] Add bootstrap intervals grouped by pair_id.

## Run evidence

This report was generated locally from saved per-response scores. threshold_sweep_config.json records their SHA-256 and the evaluation settings. threshold_metrics.csv contains the grid metrics; midpoint_estimates.csv and midpoint_metrics.csv contain the heuristic estimates and decisions. Accuracy values in CSV are fractions; this report shows percentages. Zero-denominator ratios are NaN. Invalid rows are excluded and counted. The completed 1.00-2.00 exports remain in the parent results folder.
